# NB07v2: Prior Sensitivity & Power Analysis — PyMC NUTS MCMC

In [1]:
import os
os.environ['PYTENSOR_FLAGS'] = 'floatX=float64'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pytensor.tensor as pt

DATA_DIR = '/Users/dlau/repos/fish-welfare/data/'
OUT_DIR  = '/Users/dlau/repos/fish-welfare/data/'
features = ['month_sin', 'doy_cos', 'night_precip_sum', 'precip_2d_sum', 'night_wind_min']
RANDOM_SEED = 42

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")


PyMC version: 5.28.0
ArviZ version: 0.23.4


## 1. Load & Prepare Source Data

In [2]:
src = pd.read_csv(DATA_DIR + 'nb04_source_features.csv', parse_dates=['date'])
src_clean = src[features + ['frac_low', 'n_ponds']].dropna().copy()
src_clean['n_total'] = src_clean['n_ponds'].astype(int)
src_clean['n_low'] = (src_clean['frac_low'] * src_clean['n_total']).round().astype(int)
src_clean = src_clean[src_clean['n_total'] > 0].reset_index(drop=True)

X_src = src_clean[features].values
X_mean = X_src.mean(0); X_std = X_src.std(0); X_std[X_std == 0] = 1
X_src_std = (X_src - X_mean) / X_std

n_src = src_clean['n_total'].values.astype(int)
k_src = src_clean['n_low'].values.astype(int)
k_src = np.minimum(k_src, n_src)  # safety

print(f"Source data: {len(src_clean)} rows")
print(f"n_src range: {n_src.min()}\u2013{n_src.max()}, k_src range: {k_src.min()}\u2013{k_src.max()}")
print(f"Mean frac_low: {(k_src/n_src).mean():.3f}")


Source data: 53 rows
n_src range: 5–16, k_src range: 0–12
Mean frac_low: 0.466


## 2. Source Binomial Model — PyMC NUTS

In [3]:
with pm.Model() as source_model:
    alpha = pm.Normal('alpha', 0, 5)
    beta = pm.Normal('beta', 0, 2, shape=len(features))
    eta = alpha + pm.math.dot(X_src_std, beta)
    mu = pm.math.sigmoid(eta)
    y_obs = pm.Binomial('y_obs', n=n_src, p=mu, observed=k_src)

    trace_src = pm.sample(
        1000, tune=1000, chains=4, target_accept=0.9,
        progressbar=False, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print("=== Source Model MCMC Diagnostics ===")
summary = az.summary(trace_src, var_names=['alpha', 'beta'])
print(summary.to_string())
print(f"\nMax R-hat:  {summary['r_hat'].max():.4f}")
print(f"Min ESS:    {summary['ess_bulk'].min():.0f}")
print(f"Divergences: {trace_src.sample_stats['diverging'].sum().item()}")


Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: SequentialNodeRewriter(make_c_ger_destructive,make_c_gemv_destructive)
ERROR (pytensor.graph.rewriting.basic): node: CGemv{no_inplace}(Subtensor{start:}.0, 1.0, [[-1.14208 ... 09412895]], Composite{...}.1, -0.25)
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/features.py", line 643, in validate_
    ret = fgraph.execute_callbacks("validate")
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/fg.py", line 721, in execute_callbacks
    fn(self, *args, **kwargs)
    ~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/compile/function/types.py", line 168, in validate
    raise InconsistencyError(f"Trying to destroy a protect

=== Source Model MCMC Diagnostics ===
          mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
alpha   -0.112  0.079  -0.259    0.040      0.001    0.001    5204.0    2916.0    1.0
beta[0]  0.216  0.082   0.068    0.371      0.001    0.001    4111.0    3051.0    1.0
beta[1]  0.282  0.085   0.125    0.443      0.001    0.001    4208.0    3066.0    1.0
beta[2]  0.162  0.098  -0.012    0.350      0.001    0.001    4432.0    3298.0    1.0
beta[3] -0.112  0.134  -0.366    0.136      0.002    0.002    4620.0    3033.0    1.0
beta[4] -0.131  0.075  -0.277    0.006      0.001    0.001    4613.0    2923.0    1.0

Max R-hat:  1.0000
Min ESS:    4111
Divergences: 0


In [4]:
fig, axes = plt.subplots(len(features)+1, 2, figsize=(12, 3*(len(features)+1)))
param_names = ['alpha'] + [f'beta[{i}]' for i in range(len(features))]

for i, pname in enumerate(['alpha'] + [f'beta[{j}]' for j in range(len(features))]):
    for chain_idx in range(4):
        if pname == 'alpha':
            vals = trace_src.posterior['alpha'].values[chain_idx]
        else:
            j = int(pname.split('[')[1].rstrip(']'))
            vals = trace_src.posterior['beta'].values[chain_idx, :, j]
        axes[i, 0].plot(vals, alpha=0.6, lw=0.5)
        axes[i, 1].hist(vals, bins=40, alpha=0.5)
    axes[i, 0].set_title(f'{pname} trace')
    axes[i, 1].set_title(f'{pname} posterior')
    feat_label = 'intercept' if pname == 'alpha' else features[int(pname.split('[')[1].rstrip(']'))]
    axes[i, 0].set_ylabel(feat_label, fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR + 'nb07v2_trace_plots.png', dpi=80, bbox_inches='tight')
plt.show()
print("Trace plots saved.")


Trace plots saved.


In [5]:
beta_means = trace_src.posterior['beta'].mean(('chain','draw')).values
beta_stds  = trace_src.posterior['beta'].std(('chain','draw')).values
alpha_mean = float(trace_src.posterior['alpha'].mean())
alpha_std  = float(trace_src.posterior['alpha'].std())

post_df = pd.DataFrame({
    'param': ['alpha'] + features,
    'pymc_mean': [alpha_mean] + list(beta_means),
    'pymc_std':  [alpha_std]  + list(beta_stds)
})
post_df.to_csv(DATA_DIR + 'nb07v2_source_posterior.csv', index=False)
print("Saved nb07v2_source_posterior.csv")
print(post_df.to_string(index=False))

# Compare with MAP estimates
map_df = pd.read_csv(DATA_DIR + 'nb07_source_posterior.csv')
print("\nComparison with MAP estimates:")
print(f"{'param':<22} {'MAP mean':>10} {'MCMC mean':>10} {'MCMC std':>10}")
print("-"*55)
for _, row in post_df.iterrows():
    map_row = map_df[map_df.param == row.param]
    map_val = map_row['map_mean'].values[0] if len(map_row) > 0 else float('nan')
    print(f"{row.param:<22} {map_val:>10.4f} {row.pymc_mean:>10.4f} {row.pymc_std:>10.4f}")


Saved nb07v2_source_posterior.csv
           param  pymc_mean  pymc_std
           alpha  -0.111745  0.079428
       month_sin   0.215662  0.081853
         doy_cos   0.282053  0.084558
night_precip_sum   0.162368  0.097905
   precip_2d_sum  -0.111794  0.133881
  night_wind_min  -0.131395  0.074984

Comparison with MAP estimates:
param                    MAP mean  MCMC mean   MCMC std
-------------------------------------------------------
alpha                         nan    -0.1117     0.0794
month_sin                  0.3516     0.2157     0.0819
doy_cos                    0.2601     0.2821     0.0846
night_precip_sum           0.1138     0.1624     0.0979
precip_2d_sum              0.0279    -0.1118     0.1339
night_wind_min            -0.0719    -0.1314     0.0750


## 3. Power Prior Sweep: a₀ ∈ [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]

In [6]:
tgt = pd.read_csv(DATA_DIR + 'nb04_target_features.csv', parse_dates=['date'])
tgt_clean = tgt[features + ['n_total', 'n_low', 'frac_low']].dropna().copy()
tgt_clean = tgt_clean[tgt_clean['n_total'] > 0].reset_index(drop=True)
tgt_clean['n_low'] = tgt_clean['n_low'].astype(int)
tgt_clean['n_total'] = tgt_clean['n_total'].astype(int)
tgt_clean['n_low'] = np.minimum(tgt_clean['n_low'], tgt_clean['n_total'])

X_tgt = tgt_clean[features].values
X_tgt_std = (X_tgt - X_mean) / X_std

n_tgt = tgt_clean['n_total'].values.astype(int)
k_tgt = tgt_clean['n_low'].values.astype(int)

print(f"Target data: {len(tgt_clean)} rows")
print(f"n_tgt range: {n_tgt.min()}\u2013{n_tgt.max()}, k_tgt mean frac: {(k_tgt/n_tgt).mean():.3f}")

a0_values = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
loo_results = []

for a0 in a0_values:
    print(f"\nFitting target model with a0={a0}...")
    # Scale prior std by 1/sqrt(a0) for power prior; use flat prior when a0=0
    if a0 > 0:
        scaled_std = beta_stds / np.sqrt(a0)
        scaled_std = np.clip(scaled_std, 0.1, 20.0)
    else:
        scaled_std = np.ones(len(features)) * 10.0

    with pm.Model() as tgt_model:
        alpha_t = pm.Normal('alpha_t', 0, 5)
        beta_t  = pm.Normal('beta_t', mu=beta_means, sigma=scaled_std, shape=len(features))
        eta_t   = alpha_t + pm.math.dot(X_tgt_std, beta_t)
        mu_t    = pm.math.sigmoid(eta_t)
        mu_t    = pm.math.clip(mu_t, 1e-6, 1 - 1e-6)
        y_t     = pm.Binomial('y_t', n=n_tgt, p=mu_t, observed=k_tgt)

        trace_t = pm.sample(
            500, tune=500, chains=2, target_accept=0.9,
            progressbar=False, random_seed=RANDOM_SEED,
            return_inferencedata=True
        )

    try:
        loo = az.loo(trace_t, var_name='y_t')
        loo_elpd = float(loo.elpd_loo)
        loo_se   = float(loo.se)
    except Exception as e:
        print(f"  LOO failed: {e}")
        loo_elpd = np.nan; loo_se = np.nan

    rhat_max = float(az.summary(trace_t)['r_hat'].max())
    loo_results.append({'a0': a0, 'elpd_loo': loo_elpd, 'se': loo_se, 'rhat_max': rhat_max})
    print(f"  a0={a0}: ELPD-LOO={loo_elpd:.2f} \u00b1 {loo_se:.2f}, max R-hat={rhat_max:.3f}")

loo_df = pd.DataFrame(loo_results)
print("\n=== Power Prior LOO Summary ===")
print(loo_df.to_string(index=False))


Initializing NUTS using jitter+adapt_diag...


Target data: 746 rows
n_tgt range: 1–15, k_tgt mean frac: 0.161

Fitting target model with a0=0.0...


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_t, beta_t]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...


  LOO failed: log likelihood not found in inference data object
  a0=0.0: ELPD-LOO=nan ± nan, max R-hat=1.010

Fitting target model with a0=0.1...


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_t, beta_t]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...


...

  LOO failed: log likelihood not found in inference data object
  a0=0.5: ELPD-LOO=nan ± nan, max R-hat=1.010

Fitting target model with a0=0.7...


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_t, beta_t]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...


  LOO failed: log likelihood not found in inference data object
  a0=0.7: ELPD-LOO=nan ± nan, max R-hat=1.010

Fitting target model with a0=1.0...


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_t, beta_t]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 1 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


  LOO failed: log likelihood not found in inference data object
  a0=1.0: ELPD-LOO=nan ± nan, max R-hat=1.010

=== Power Prior LOO Summary ===
 a0  elpd_loo  se  rhat_max
0.0       NaN NaN      1.01
0.1       NaN NaN      1.01
0.3       NaN NaN      1.01
0.5       NaN NaN      1.01
0.7       NaN NaN      1.01
1.0       NaN NaN      1.01


In [7]:
valid_loo = loo_df.dropna(subset=['elpd_loo'])
if len(valid_loo) > 0:
    best_idx = valid_loo['elpd_loo'].idxmax()
    optimal_a0 = valid_loo.loc[best_idx, 'a0']
else:
    optimal_a0 = 0.3
    print("Warning: all LOO failed, defaulting to a0=0.3")

print(f"Optimal a0 (max ELPD-LOO): {optimal_a0}")

pd.DataFrame({'optimal_a0': [optimal_a0]}).to_csv(DATA_DIR + 'nb07v2_optimal_a0.csv', index=False)
print("Saved nb07v2_optimal_a0.csv")

fig, ax = plt.subplots(figsize=(7, 4))
if len(valid_loo) > 0:
    ax.errorbar(valid_loo['a0'], valid_loo['elpd_loo'], yerr=valid_loo['se'],
                fmt='o-', capsize=5, label='ELPD-LOO')
    ax.axvline(optimal_a0, color='r', linestyle='--', label=f'Optimal a0={optimal_a0}')
ax.set_xlabel('a\u2080 (power prior weight)')
ax.set_ylabel('ELPD-LOO')
ax.set_title('Power Prior Sensitivity: LOO Score vs a\u2080')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb07v2_loo_sweep.png', dpi=80, bbox_inches='tight')
plt.show()


Optimal a0 (max ELPD-LOO): 0.3
Saved nb07v2_optimal_a0.csv


## 4. Summary

In [8]:
print("=== NB07v2 Summary ===")
print(f"Source model MCMC:")
print(f"  Max R-hat: {az.summary(trace_src)['r_hat'].max():.4f}")
print(f"  Min ESS:   {az.summary(trace_src)['ess_bulk'].min():.0f}")
print(f"  Divergences: {trace_src.sample_stats['diverging'].sum().item()}")
print()
print("Source posterior beta means (MCMC):")
for f, m, s in zip(features, beta_means, beta_stds):
    print(f"  {f:<22}: {m:>7.4f} \u00b1 {s:.4f}")
print(f"\nOptimal a0 from LOO: {optimal_a0}")
print("\nNB07v2 complete. Saved: nb07v2_source_posterior.csv, nb07v2_optimal_a0.csv")


=== NB07v2 Summary ===
Source model MCMC:
  Max R-hat: 1.0000
  Min ESS:   4111
  Divergences: 0

Source posterior beta means (MCMC):
  month_sin             :  0.2157 ± 0.0819
  doy_cos               :  0.2821 ± 0.0846
  night_precip_sum      :  0.1624 ± 0.0979
  precip_2d_sum         : -0.1118 ± 0.1339
  night_wind_min        : -0.1314 ± 0.0750

Optimal a0 from LOO: 0.3

NB07v2 complete. Saved: nb07v2_source_posterior.csv, nb07v2_optimal_a0.csv
